# Test `twitter-dataset`

## 1. Setup

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
import rioda_python_driver as riopy
import crisis_image_benchmarks_classifier as cibClassifier
import crisis_image_benchmarks_middleware as cibMiddleware
import twitter_dataset as td
import matplotlib.pyplot as plt

## 2. Test functions

In [ ]:
# load twitter dataset from .json
tweets = td.load_tweets_from_json(dataset = '2018 Aude Flood')
print('Count of Tweets :', len(tweets))

In [ ]:
# Fetch images from a single tweet
imgs = td.fetch_imgs(tweets[24], online = True)
for img in imgs:
    plt.figure()
    plt.imshow(img)

In [ ]:
# Fetch images' absolute paths from a single tweet
paths = td.fetch_imgs_paths(tweets[24], online = False, folderPath = '../resources/twitter-datasets/2018-aude-flood/2018-aude-flood-images/')
paths

In [ ]:
# Load pretrained models
pretrain = cibClassifier.load_models(verbose = False)

In [ ]:
# Classify a single tweet if images exist with cib classifier
results = td.cib_classify_tweet(
    tweets[24],
    pretrain = pretrain,
    online = False,
    folderPath = '../resources/twitter-datasets/2018-aude-flood/2018-aude-flood-images/'
)
results

In [ ]:
# Translate the classification results of a single tweet with cib middleware
concepts = td.cib_translate_tweet(tweets[24], results)
concepts

In [ ]:
# initialization
uri = 'neo4j://localhost:7687'
userName = 'neo4j'
password = 'neo4j'
driver = riopy.initialize(uri, userName, password)

In [ ]:
# get the name of current knowledge space and collaboration
with driver.session() as session:
    currentKnowledgeSpaceName = session.read_transaction(riopy.get_current_knowledge_space_name)
    currentCollaborationName = session.read_transaction(riopy.get_current_collaboration_name)
print('The Name of Current Knowledge Space is :', currentKnowledgeSpaceName)
print('The Name of Current Collaboration is :', currentCollaborationName)
driver.close()

In [ ]:
# instantiate concepts
with driver.session() as session:
    td.cib_interpret_tweet(
        session,
        currentKnowledgeSpaceName,
        currentCollaborationName,
        tweets[24],
        pretrain = pretrain,
        online = True,
        folderPath = None,
        dataset = None
    )
driver.close()